# Caso Challenger — Regressão Logística Binária

**Pergunta de pesquisa:** Os dados disponíveis antes de 28/01/1986 permitiriam prever a falha dos O-rings do ônibus espacial Challenger?

**Técnica:** Regressão Logística Binária (Supervised ML para variáveis dependentes qualitativas).

**Dataset:** 23 missões anteriores ao acidente, com observações de temperatura no lançamento (°F), pressão de verificação (psi) e ocorrência de stress térmico nos O-rings.

---

## Contexto histórico

Em 28 de janeiro de 1986, o ônibus espacial Challenger se desintegrou 73 segundos após o lançamento, matando os 7 tripulantes. A causa raiz: falha dos O-rings (anéis de vedação) dos foguetes laterais, que perderam elasticidade pela baixa temperatura ambiente do dia (≈ 2°C / 36°F).

Os engenheiros da Morton Thiokol (fabricante dos foguetes) alertaram a NASA na noite anterior — disseram explicitamente que os O-rings nunca foram testados abaixo de 53°F (12°C). Foram voto vencido.

A Comissão Rogers, que investigou o acidente, concluiu que **os dados pré-existentes já apontavam o risco**. Este notebook demonstra exatamente isso, ajustando um modelo logístico que poderia ter sido feito com as ferramentas estatísticas disponíveis em 1986.

## 1. Importação de pacotes

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from statstests.process import stepwise

## 2. Carregamento da base

**Variáveis:**
- `desgaste` — quantidade de O-rings com stress térmico observado (0, 1 ou 2)
- `temperatura` — temperatura no lançamento em °F
- `pressão` — pressão de verificação de vazamento (psi)
- `t` — id do teste

In [ ]:
df = pd.read_csv('../data/challenger.csv')
df.info()
df.describe()

## 3. Engenharia da variável dependente

Como o dataset não tem variável binária direta, criamos `falha`:
- `falha = 1` se houve qualquer stress térmico (`desgaste > 0`)
- `falha = 0` caso contrário

In [ ]:
df['falha'] = (df['desgaste'] != 0).astype('int64')
df.head(10)

## 4. Análise exploratória

Pairplot para investigar separabilidade entre as classes em função das variáveis explicativas.

In [ ]:
cores = {0: 'springgreen', 1: 'darkorchid'}
g = sns.pairplot(df[['falha', 'temperatura', 'pressão']],
                 hue='falha', palette=cores)
g.fig.set_size_inches(8, 6)
plt.show()

**Observação visual:** as classes se separam com clareza no eixo da temperatura — voos com falha concentram-se em temperaturas mais baixas. Pressão não mostra padrão evidente de separação.

## 5. Modelo logístico binário (modelo completo)

Estimação por Maximum Likelihood com `temperatura` e `pressão` como explicativas.

**Por que Logit e não OLS?**

OLS modela a média condicional de Y quantitativo. Quando Y é qualitativa (binária), precisamos modelar a **probabilidade de ocorrência** — e a função logística garante saída no intervalo [0, 1]:

$$ P(falha = 1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 \cdot X_1 + \dots)}} $$

In [ ]:
modelo_completo = sm.Logit.from_formula(
    'falha ~ temperatura + pressão', df
).fit()
print(modelo_completo.summary())

**Interpretação:** `pressão` apresenta p-valor ≈ 0,538 — não significativa ao nível de 5%. Vamos aplicar Stepwise para selecionar variáveis.

## 6. Procedimento Stepwise

Seleção bidirecional de variáveis baseada em p-valor (limite = 0,05).

**Como funciona:** o algoritmo itera entre adicionar/remover variáveis até que todas as restantes tenham p-valor ≤ 0,05.

**Limitações:** método guloso (não garante o melhor subconjunto global). Em datasets com muitas variáveis correlacionadas, pode gerar modelos instáveis. Recomenda-se complementar com VIF e validação cruzada.

In [ ]:
modelo_final = stepwise(modelo_completo, pvalue_limit=0.05)

**Modelo final:** `falha ~ temperatura`

| Métrica | Valor |
|---|---|
| Intercept | 23,7750 |
| Coef. temperatura | −0,3667 (p-valor = 0,036) |
| Pseudo R² | 0,4536 |
| LLR p-valor | 0,00054 |

**Interpretação do coeficiente:** o sinal negativo confirma a hipótese — quanto **menor** a temperatura, **maior** o log-odds de falha.

## 7. Predições

Avaliação da probabilidade de falha em três cenários:

In [ ]:
for t in [77, 70, 36]:
    p = modelo_final.predict(pd.DataFrame({'temperatura': [t]})).iloc[0]
    print(f'T = {t}°F → P(falha) = {p:.6f}  ({p*100:.4f}%)')

**Resultado-chave:** nas condições reais do lançamento (36°F / 2°C), o modelo estima probabilidade de falha de ~99,99% — praticamente certeza estatística.

Esse modelo poderia ter sido construído em 1986 com as ferramentas estatísticas disponíveis na época. A teoria existia (Berkson, 1944; Cox, 1958) e os dados estavam tabulados na própria NASA.

## 8. Visualização final — Sigmoide

Gera a figura usada na divulgação do post.

In [ ]:
import sys
sys.path.insert(0, '../src')
from visualization import gerar_sigmoide

df['phat'] = modelo_final.predict()

INTERCEPT = modelo_final.params['Intercept']
COEF_TEMP = modelo_final.params["Q('temperatura')"]

path = gerar_sigmoide(
    df, INTERCEPT, COEF_TEMP,
    output_path='../outputs/challenger_sigmoide_linkedin.png',
    pseudo_r2=modelo_final.prsquared,
    p_valor=modelo_final.pvalues["Q('temperatura')"],
)
print(f'✓ Imagem salva em: {path}')

## 9. Conclusão

Os dados disponíveis nas 23 missões anteriores ao acidente, modelados via Regressão Logística Binária, indicariam:

- **A 25°C (77°F):** P(falha) = 1,1% — operação dentro de zona segura.
- **A 21°C (70°F):** P(falha) = 13,1% — risco perceptível, decisão de borderline.
- **A 2°C (36°F, dia do acidente):** P(falha) ≈ 100% — risco proibitivo.

O caso Challenger é referência na história da estatística aplicada justamente porque demonstra: **não basta ter os dados — é preciso modelá-los e ouvi-los**.

---

## Referências

- Dalal, S. R., Fowlkes, E. B., & Hoadley, B. (1989). Risk analysis of the space shuttle: Pre-Challenger prediction of failure. *Journal of the American Statistical Association*, 84(408), 945–957.
- Report of the Presidential Commission on the Space Shuttle Challenger Accident (Rogers Commission). 6 de junho de 1986.
- Fávero, L. P., & Belfiore, P. (2024). *Manual de Análise de Dados* — referência metodológica das técnicas aplicadas.